# 🎙️ 會議記錄與摘要系統（Colab 免費 GPU 版）

上傳會議錄音，系統會自動：
1. **辨識每位講者**（說話者分離）
2. **產生逐字稿**：使用聯發科 **Breeze-ASR-25**，專為**台灣華語、中英夾雜**最佳化
3. **整理成專業會議記錄**：摘要、討論事項、決議、待辦事項（負責人／期限）、各人發言重點

錄音可以放在 **Google 雲端硬碟的任何資料夾**，逐字稿與會議記錄會**自動存到錄音所在的同一個資料夾**，關掉筆記本也不會消失。

---
### 📌 使用方式
| | 步驟 | 說明 |
|---|---|---|
| ⓪ | **開啟 GPU**（只需設定一次） | 上方選單「**執行階段 → 變更執行階段類型**」→ 選 **T4 GPU** → 儲存 |
| ① | 執行「步驟 1」 | 點儲存格左邊的 ▶️。授權 Google 雲端硬碟時，**請勾選全部權限**再按繼續。約 3～6 分鐘 |
| ② | 執行「步驟 2」 | 在出現的資料夾清單中點選錄音檔（或從電腦上傳） |
| ③ | 填寫「步驟 3」並執行 | 填會議名稱、講者人數等，開始處理。1 小時會議約 10～20 分鐘 |
| ④ | （選用）執行「步驟 4」 | 聽各講者的聲音片段，填上真實姓名 |

> 💡 **每次重新開啟筆記本**都要先執行步驟 1（Colab 每次都是全新的環境）。<br>
> 💡 處理期間請保持此分頁開啟；閒置太久 Colab 可能會中斷連線。<br>
> 🔒 錄音會在 Google 的伺服器上處理並存在您的雲端硬碟，極機密會議請自行評估。<br>
> ⚠️ AI 產生的會議記錄僅供參考，正式發布前請人工覆核。

In [ ]:
#@title 步驟 1：準備環境（每次開啟都要執行一次）
import os, sys, io, glob, time, base64, tarfile, subprocess, ctypes, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
_t0 = time.time()

# ---- 檢查 GPU ----
import torch
GPU = torch.cuda.is_available()
if GPU:
    print(f"✅ 已啟用 GPU：{torch.cuda.get_device_name(0)}")
else:
    print("⚠️ 目前沒有 GPU！請到上方選單「執行階段 → 變更執行階段類型」選擇「T4 GPU」，再重新執行此步驟。")
    print("   （沒有 GPU 也能使用，但速度慢很多，且摘要改用較小的模型，品質較差）")

# ---- 連接 Google 雲端硬碟 ----
from google.colab import drive
USE_DRIVE = True
try:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")
    print("📁 已連接 Google 雲端硬碟")
except Exception as _e:
    USE_DRIVE = False
    DRIVE_ROOT = Path("/content/上傳的錄音")
    DRIVE_ROOT.mkdir(exist_ok=True)
    print(f"\n⚠️ 無法連接 Google 雲端硬碟（{_e}）")
    print("   常見原因：授權頁面沒有勾選全部權限，或彈出式視窗被瀏覽器封鎖。")
    print("   解決：重新執行此步驟，授權時點「全部選取」勾選所有權限再按「繼續」。")
    print("   👉 這次先改用 Colab 暫存空間繼續：錄音只能從電腦上傳，結果會自動下載到您的電腦，")
    print("      關閉筆記本後暫存空間會清除。\n")

# ---- 安裝套件 ----
print("📦 安裝套件中…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faster-whisper", "sherpa-onnx",
                "opencc-python-reimplemented", "python-docx"], check=True)

# ---- 解開系統程式碼 ----
APP_DIR = Path("/content/meeting_notes")
APP_DIR.mkdir(exist_ok=True)
_CODE = "H4sIAJma/moC/+19+3cTR7Lw/qy/YjJ8OYwceZD8IKBFsA6PJOcSwgWyd+8RXmUsjWxdS6PZmZFt1vgcO4mxDQY74RFeCZAQQgwG50EwxsD/svHo8RP3T/iq+jHTI8mGZG+y33ejOQke9fSjuqq6uqq6ulvdom7500Ft5C1dy+jWH36VJ0qf9f5Go51d/jumx6IdsY4/SCN/+A2eku1oFjT/h9/n07FNKji5gp6Ivb69qzMajXV2qR3RbiBJZ+gPred//aOZ5pZUKmfknFRKNY/9auN/a1fX+uMf3mPdHa93dXd3dXZthfHf0Y3jP/pbjn+rWHQ2yvei7/+fPur/E/K/s1H+x1ry/zeR/6/78r87unXb1th2taO7u7OroyX+fy/yP100srn+X0v6v1D+d3Z1dRD5/3p3LNYZ60T539HR2ZL/v8Ujy7I7edu9dq16e9G9d/n56mV37n7l3O3KJ0vuFx9W750sn1+u3jrh3r/z0/gHkDmUK5hFy5GKdihrFQuSqTkD+VyfxJIPws9Q6I2ew3tTe94+JCVIgpJKZXN5PZUKq5ZuF/NDuhJWTc3SDYf9Cb3z7p69+w+LZYq22q87ujGkyO8cSPnf5YjkVb9FkgvFjJ635XA4tKfnSM/6FfCvdcUzmqNh4dAmqbrwee3a99Wnt6uLFwEN7+iZnHZEH5TesHT973p7z+FD7R3dz1en/2MgZ5u6JeU1q19vH+qQalOfuUun3bmlysSX1bn7UM+WteXF6qlvyw8fAibdp/eqC8+er86EoAraEQAxCJ33BcCzHUsR0AFA9lEINNsCCNrTTgdCjEX27P3z27v3Nq2NfoLqZK3kFOWw5D2bJEyBetNmCf8tZTRS2e533zn43pHmtbFvWF3OcLYJ1W2SMAHqyeaLmhPbCm+YkGI/Sc1v7O15B6qFdKWxavyI9XZjp3YffC915K1De3v2HG5agH1jWIJv0IdUuliCjGGpaEldYU7Lheo331bHJ93pE7UrXwA5zWOaYRQdXbL1/gJwnObkikZ7pxqVXpM697QfNnVtEIi695BudxzQnT93AKWBiuULU0i5PW/3HEod3vvmOuTjn5tSD9jIkvFFbFktGsYI4VosuvedNzaqGT5vXLNe6NMzmZzRH6wWsXX4rXf374FqCT2UJpV7mZAIURVoG0aqVn886d78WvrHiU+k6t0rgElMWPoYkVu+fan2+PPaBx+5Z64BZncfaT9iaYadLVoF3aJoA+ZHtB1878DuI+v0jHxr2i2zZKQd2RvcXp+w6Y8vVm9NSPv3vwPtvPnme/uer87m81pBA0YwpfLVu+7VpfKZ29U7F7B5yLZO496XpgDk8wXS/N+GdaOzvauvPWdAplLageEXfb39b12pwVRB7e8vZREsrGv3kb805Vf2DVEb29q5rcvL/9Z7B/4N/u05dHj9cn4eMj5gpmS0qZ1/WL76YXVxCbi7fO8BRUrl8kfl+3Pw0138FGQ2aeZNGE77e/5z7wat+FkI/RHATZL7dLZy5z7QluAWUOtjG3Bc/uaaBAMVMFyeviA5Pu1tyPZWqb8fGFHap6V14JQb7uenoFRt4ht3flraXcxrfdKbpCwB8I2e3f+298CepvRh3xAsDoZMsffWvtTbzcuQL1ji34F0W/6d0u+N9rdF+kEl3lyhFgYzOUuhk5CdOGKV9Iikj+RsJ1UcJD/D/8vVYLXl/2v5/3z7b3t021a1oxOoEOtqGYC/E/vPzJl6Pmfo/xr/Hwz1rVFu/4HdR/1/W7e27L/fyP5Dq+f2SarqBfVmklYbnweVonIbTJhpd/4O1QZBB6zeBjNphlmFxBYEM6/klCww9Lg5SHRuovDaIW45Wjp/s0t9plVM67bNU5wBC+Yh0B9ohWidpfOabes2r9FLikiancmlnYiUzen5DC3gHDNR+WB5d2ugOPTlda9po1Qwj0E5yTA9O9bUjXQ6RIurvCT1iIRCh9GajG0FLg2l0mlUOUh29V34s3u3ItsdzrApE6WssnQDsEWtwPLil7U75yi+WMrkl2AKVr/5HDUf0LAqD25J3JBEzW1uvnJzhXy+7K6ed5fOVFeX3akVsTpqjVSnp9Bef/IF1cKpqX3wENhnRwA8ee3xV2vLp8oX768tj7vXf6DlsQGiL9ZmPwJigz7mTn7t3nu09uQZWvqPJrza8dPNp7UrV9F6vTDlLk2Xr86486cBMiR0KHTQKvaDDW/vM6A1jt9kEtSrCDUxeiPSgaKh94ZCoT95tAqRf6X3HEcHbTGtx0NoOtrU5oqjBk4TYCQ6cVoPSdCNjPjT0Uccmpv8zOhZySmmkAkUW89nw7RafCwd+NBgHEI/NgOI2C1pK2c6tGimZBFmFRvNa0Z/SevXfTBLvBt2XMqDqpj0+tWLhhZyowKwaaW8k8pqaadoHUtgvvDLgj0qc0BkaBWyqPw36sIMHv6J/46EpLpH9gGFzMmSylsNS6CvSyWwCGgVfr7eMQrjn2wctemC7gwUMx7UOEZoDZlGoH1cKpmk3wHghwxq6YoPOHTi7wNyuBFg//FRqrS1lQR4oWqhV729SFYEDciVSWmlTK6ooEeKECsste+Eka4aGc2ytGMUYhBWwPNSNlsw9X7JXZpae/x47cn58vVVd3UORmzB7NxS6NK2DGtDWwpm1z/Gb8GIrT6ZqXywAoJgUHIv3KtOfFebOEtZpLODCUCsO13IAAMkZVo59rPdKNoOyDPyPoT/6pZVtMjPHPyLsOJ7FlOynR15wI6IFrldSxPjkZTQLGauHj4Uxt9yL8lbLDnQrC9MVatkKABLREprJpHHkMMsOcy8SQ/oaWbbqAAcfAsJZAR8IZn7StmsbinwEegHUlVP4Afa47CaLprHFET9Jqndezxrb9p9/GNt4rvq6ow7+dVP4xMgryq3T7mT3wLiAZlCkVAqX0wPAvCe3Ff3QwJUndJsC9JRlIRS6OeAH6NjoRTxDfAPlPTAXDRZYUzZny/2aXmJJpKU4ZwzIJG2fLbNZVkOKWeT6uIBfvTdnE1S7QHdMrUUOiZwNrGLwUxZCWY9tE6RvCoxJ22FOdkFl0g43jACGBFI78R0r992UX03m0Vl7SCmlMggU5qm7iYNKg1tEKdKommRd/DTeuUIhzspwd5PNPYpgnNsipLT5t8Frx7654SOMhqFQqm3eg5TlxB00tKBwwpmLq8rlpyEOQnG2PPVieer14Cbnq9egUkyor6yq1fmo99kXdAVb4oggx/+eqPed+NK7vIynV4rH92g0yjOdYufQtXVU1+V7z2ozj+prK7QT+7MFPAt5e3ql1fXlk8KI94EaAX+CzHym5yn0CeZ1w3F756azYE8yucJpOGw1CZ1dEs7EyQXSWoQrJhKcaYNU+TAUAfEHLVfU3YlkkdLXXo02n60tD2bzfaGjyu7djQkQlYUIPA/aUKQGqaqZTIpU+AlaCVAIsinWrqZ10AWyxGsBggih/20XTTtmpj2Ck2bANUI0IFN6Xlbp13xRy2M8foxC0kbjlgUC03HK9HfspoNc0NqmClWbLQyPYswd6CMYx1rHIJM8oiF+Nj1vPQgE/WhXFpPCB+ow32jSU0YRsDdJUdPEcEq1MHc7C9ZiVnaaKAF6tBH0rrpSHvJH6AySi3QaEBtXXt6yr31Afrktux+b0+P5E49gYnQXbmztny6cmUZtERiBJTPPcJpExpogM200LOYlZPVxVvuxyd7JRxV358HTVQabcDP2JbRxv6OSXSuoCsw3jQyqo/BdIE+R69xOfzP0UsGpAFrBvBPVzT+53DORRuyMuX1kkHVE5/d0eqaOlU+91TsMhhU5RM/UsOBekpB2FTvnQVBhfRBm4G6ejmGZgRJVD+AxAlUmLn60zR3GiRsPq8ToeUPSJxoc3+HeR+FOFPPQckFBEnHSVUM/A2nwUH9GDQs1oBSsD220bjGMjhngoZHgAgOy3S2PzD1scWaPQTaDec6cckl0VDBYeHrC6c+voC0cTUHWa4XVufPxIyPgktM62jHL5hcGwt5K0MINoN3L0/bOwKTOdonGwHaCKS3WhX5+fCk8yUU0QygfSCvd3spG0GBDbGydgK4JUJURnugmM8EYPNWs5q0XcgZKW6XANMmomrnizJls5CrO5gr3EzZAxZVh7R8LoNKSDPFTsvB/HeoZKCvdS/aAIpc52zhQgCX4Wuz96sLN0H0Ve+cqnyxUl04Tb+WF86Vb0y7ixfdq7frZCEZN0lATe8LxooCsAYllVeyXqN35z8qL35V++aaqLOjsGCCQiEWV1wwsiLSeqIDjB3mO4hLvheBocrOMEWqqQQSDOd0n5IhlTlFR8sLiOaVN6B1bXkRJD4WkrZIBW1EISUjUiwcrte0oiGGFhtMd8RiRmUmFe0oGlT5fJ8G5lO6D4wnkIGpvmMp4rZIIWGZBghSx0ZDULFU8i0iWapuZPAP6xK1aC2Ud7Q1asnhdPxZ7daF8t0bqJ3OPa1dP+GuzNWmTpcvLFUe3q5ewgVX6oGTYqAMs9cOMFOpUmdldGYqEaUIGklF8D+bGvv9to8xkle1dYf5KhQ7QrRQkh6WXgMMiVySVLSI1BeRsjJrdJRkTNq9YzLtDv3utdTLJhaHOQb6mrLLQNEZhopsorRvyCMBbY2zrlC5p1FGJEYtXrdfLe0R04bE0UgVIq/6+tmUjXSiPpD+gREO7WkO/MVMEbZWLVPvV1jSjAwprdIlbKp+1MkFIhNCfiAFzvHutYfVG7PlSx+4N78tn/8UjWnQzlAz25Iu7TlwQKrMTJev3kXl7C56JoK6mQScUn34caheNTtqeMoZNiJGunjtNNG2JMp4PCRmtjZ+zV25ha7E1Q/Lk18IAujnat+/idL2sgpyE16iNN2AjShri0UIoVmZX8ri8KlgouQRvbqvSUBB0RMLImCU1zmGKiCyJk+ghpYse5IIVRMAPWdki1AxZUcBbA8LrLPcS5dAHx2IWc0eTMh+fkjq0zUQzSCkRcsFo3h8aiAkRCCC+CuYfF19CHTgbC4PPEITvOz4wdQsraCT6X1U9uZ2OS7BLN2NwwvmZTsH8imt+/NzAR2bsWg0uj4jyMBgenoA6s/Q7Fuj0TE/Owa+57R8imI+Qf/4JEt4aAURQWaxkG/DGZkcUyWgOLBqsWSn0MJN7NOABhESqXHp9NqTM7XzD1HJXzj30/gsjtCbU+6jx9VbKz+Nnw75GgXvlUSdAYQeYZizDh8KcZySWaXXE+1AXSZsCZGbT4b+UKczISBSgRI4IUHlvmM5pkaFGRE4CjNRvg0glySp0E2oQFGG+Qw3TGe4YVKEzgfDDDhai183cmizOjXTJHViEVYrA5S+UI+JOCexYYm87fnB+dBkM21KcxQnCowcI5XgHAdiyjfDyjNPYaqtTk9JScjmxHpR6FUunCxfHXdvXq5c/ojOds9Xr4BZSz+5M1NQCjJUn30MGQQjrA8YPkL+TRWHmPUVAQ6OeiSrmyMDRpAm7QQwg5jpA/E16E/ZWCkSsA87FJbaiUIDdTrRAOkg304ORl19dRACPopD3G2FyQ1TLCaGBE03CDbLhDjFIcJ1gxjFRyGHeh3gH4QYwrtF6hDJx1gRYADlM5HXCn0ZTRqJk3Stz1ZGktFe7GMug3M6SYjxhHA42YEaRmrvgT2e41D2HIWv7AKVSGa8AHp5v6EwZqENAt5S/ZqZiIF2T37AkEt0dESJ27BuUcfnlZMn3cVPQQKDlgZTdO3CWYz7nF7iLDLrnjjtzk+vPXlQ+WClcmm1enucOhMFFik5TtNlIzasCSIFsx15hrMvGU91I9I20X3+Amb3ZVYJZ10EIdkOmASK4judMgI+Z/iiYGZUY+AvV1ulRIK0iMlA1HbyDQXJDo7QBjmMWZFtFJRpmJ2MYmBPhnSvCUwHqOK9xAngUTVcpzTxrNJrCWm44QvCkoDubyBrsMdc1PirTNCriI9oQRAygsB3jx5kIYrQ0ctVojABJdLoWTGGdMtRfK80/QxCzcqZSp0LPFkKVkqoQvJzBRqXc7xFrfUNLE/eBPQNHBWCntRM+cAJh449VHHiTIOFcVC/PCpMK/fG3bkLNHAA5hQmfciUhbUFF+M8t7gwre2AaU2QIsQ6/rOWL3HbmFYNEyconLWJM5Vri+XpC9SxWP36y/Ln83LA1ArYo0EURYKd8/qAPxklmGTAGQS5p85eERXAF1blo6h8/geQBSx2dnrKi6KQ6VQbar50WjfzU6AijRIszFXQbAFMT1vhy+bw1WIKSHAFZACsT71AUDUESqACOTtBFaJwFHBK8j5BvgioSR7ZoMYGgZ+VR9G0GAjHox2ZsTj5URB/2PRHhPxQbOlVMCelNqKswZfOzJgc+gXVya2YtFb8Zyv+89eL/9y2vbtb7eyIxrZ1drfG2u8k/tMuFQpk/vyVAkBfGP8Z2+rFf5K9wDF4j7XiP3+j+M+1x1/RnTNsX423oeb56oy75GsuoNFU5k+AaYPOqK8WaUhh9fbF2izawN6mlPKZG+K+FLJ87D65BKX/mVjRushQtnqJmb3IUOrGxvi00EtFd9JEHvzMP/rKWIQpV6ENYz83iGHK5wteqNLh/zx8ZC/ugqMeN3ntyXUapbn25DTFpxelSVFa+fpi+coy2XFz2j11qXxhlQZzrq1OQGYSjUkqqt45hbGeT0/RaM7mEaNiDOrC57hHiyz742LC42uQE5cSvp8CjRvDtc7C+wK+AFg3puGl8uN31Yf3qXeYt7rweXkJm3FX5zDHd08rX6zAS/mLz4BHkP4XzwCM7txC+fqj8tnLHhPRyFPKSlDd2vJpd+5++fRcbfx6+eoCXfVArwuB5vnqFdaeX37ufvXDJ+WrM6JnqzZ7n4Tv4HIZ2OZryycxCPbCFCjg0Mzas3sAKECztrLizp+m2Cp/h9jyqgVUeb4eyFO5/S3BMGjZdBuWH2Er+zG2LKiW1wEVVO7elUZzY1tGjTEJRgAgmgcWXk5Sj0GvxL0Flykq6DJ+CPdV3Xvg959nGuV2zFgIOuc+u+7ev1U++xCsL1aCVTKLo/iLz9zpT2loE0AI72sr52rXJ3EV4emse+UKvJe/+wQdyUvfkh29s2BS/TR+GopXHpyntKheulL++hQh9uVQVJXEZsq3Pqg+W8Lqnk26N+5ADs8TQleqyCLBPGC4urAAHFC+PwfszWygu1fWVk6sPR6vrawiuM8+q5y/5E4+BO4rn18GDCL3fAqSCIO8uKN7JhRTpertM9U7FxDTN2bRkpo6XXuMPFy9/RjbuHjfPbvgNVOem0ckAga/HifZZkId0IlvV1AyTV9wJ78Ffgl1qpL7dLL69NbayimKIP4yV/3uevW7u8Aq8A7A1C7NkzG4gBXPnaRYw5+zU+69ywR3M6EuqG7pMtRYm5ovL09C6+75OYA2FBKC5y+HRtGqH0MRGNr39oGe/QGuAurSoULZa9S20mPY8idL0CbKicdPfDlBxp0oLRCjZPxL72jWYKY4bKA76vIPwH5Aosrk15QfKneuV+6PAxIpW1JWIVs2T4OoZjHnkz+uPb5AcwIp6Cf35HV35jQ6tMAMfzBPhAjhkNCmTdLayiwSj0p/IvRDWOfjr6TO508+3yq5c1+592ZhzFNvGOMNkr1y5R5xoUIzjxECQmrKbAAltkS3cpJmlmdQTpEslFykmaefUdbA5V6k4CxwHbxQdACxNkFRmkM6ABBjVN+NWcr0REo8osOGsxUk3oI8lZsrlJXKH81Vb50ShiQFBkYT9pnwlQ8MXaOlFdJoQvRZXDwDwpFmRaowFrpbvnuDISGYhzEVxSyKZpFVQ8cl2oh0PMDC8NPjXHinnCsdDx1vb28P/I9geuzsFQGy0iIwrJswOOnx5CKiH3gHgB6fpP5NijVSpzDWgV2J9IPis21tolRta4NUWohyitjZ5Q9pZ+koov3C2evpbGVltvrsSfXRTdISQy+U+ml8HhdUgHFuzbkzUz+NfxxqlzwQ3Q8nQdYFhCjOAiDmnn5KJS3ldTov4PTB0QHzFswygIva+KW1lStMtExfqEMNCu52qTwzjiKVMC0Ixtr1aSx79TblLpjJyhcu4VQ889SdXkKBefYy8oaXTiQyMhhhUo+r8MUTjIQ2bJ6nYnpqoQri8Mur7uQkBQNDV+muYAJJZekG85qtrqw9mqMigzaBTAgTwK0PYC6h2XwhQkY76Hggq6H75cfPmMiAqeIhTJfXsa3QaF7r0/NjdTLNCyMDvac+rhOSNozrRFWpeRx2lqlramC/MIhMceOx3Bhxw/XGopUeaPhINwsJFfDsPSWnSJaj9xWt3VrJ1vL734mQ1CPFQd3A0JSGypwiqn+BPCSGHlcFoQ3QLTOK0AeyR7kxjhHPYUAtEuFV8Yeas1PakJYjW2yUcPM4btZwPcgN7TeP5q4DKkJbT9GAfwoJP9YB6YAQ0jhe/1NnR/O1V7p2nypoZkLGgnKwBuJjbt4lVR/S8k36y9RpBdAdoTnrYlwbXP0epYlFk8IjAhiZ92PCei2Qj3Q9P4X+64SAJx73lko7I2L67iN/eUEEqbFBLAJW2G+WUnntGC6AC/X62/NfUP2QbvUVbZ25fgPhXTj2AoGowvj0AlHR9pMaI1ArizMoqFbmQUSBBEIVlxwMQ2fX5iGofKyL9k9DAE8wHnXdARsIVPUHnDdG9ILpHEultfQAHyL1oc7Cqjhu/GKrw/26oVu4NFPQbVvr19mSoIMD2A56zmkvPLkW+kVSqWnnfE7GRROGNbrvLU1EwSAuVOWhfwMaGKLQ13wQZIzdZz3JkSgEEsRAoy0o9cl2rbpwi40fVs4BPBSBF2XTkSMAEAoh4OCcMYixmwKXkd6RyASAOinnDLPkpCBB7lWdohIIf/LjQ3AWoFTMGVndIlEdmLU+dJJuUqCVeCTL4SKI5gCENAgEQ1RI28G0JgC8BA6QCwx9mHFCwmcKjCFM2RrQQH9ZfCLFEOSSpSeiageKVzNlwut2JI+pOzR+xATs5p1jiZga7X6JWjGShYAEWE4gj+hF20toiGrCDBk9jbgFZCajESSWag9opp6M9cZ7I5I9mDNxtSyNcTCs2/TkC58CwJxqGqSXo1NexFisvE6iSTk3Jny23KDfoZfkQdDXHAExMXF4CkSp36mSlNMDxRzZGpiM9iZlBhP8kmG8InfIfFWVdEShQ4atrApNdES7tgXFAF/g9dgwOSpbxTzuw5TtYzb0Gfe88FbiEnX4jK3XY79wydatYFEK1FhvQCaJQHh7gHaQEblTbdu1Ywt9Ffb54I5crd9OQO7DYb74HJgZhMVqujEoGLFpgsgD7cFWAAeCX8yPkYA6WXQEX8zOyslRtiRZokE84bFeabTEIwhQZxyly9tjsr/w7VjiJlSfQiVj0FYIDHG/RWBZ0LHI0nczUGipCMYDwLxKQisi5H8/CidvYKO0XlENNaTXyCJ13sAIBWyFhyfUBSGQJngUgXzUkNX/KuZIfEM4XB+U4AMRFQMxeGloy0s2MKqBA/CaFPPmGhGAFzfOiEEzMmTyCBHc2NtAUMbhgDNcBfb9qApR8cKRQBqiPRgLEaCeD6iT9Mjeix0rkfCMdokxht+hunyIAMoijOkxYJvscC0oDkbZl4Alw+S8r5jYX4zHBlVXwfV4YIhe+q+0RWIR32n4RSPTSPwV9isYd9SO8UXhugAyO5Uz6rYV1jGcretGMDCvQAO7yV6/HCAGd+v1Sor62q4w8D/fhhdgvYLabxVLphIL840wWG2Q7TCF093PH6A5ZuEdAOobxWEjZWggBpW+YuYYk3N+9IjXEyrrdLFbGPC08EX52ldg9Da1+sEApsYhGrEX77vzt6hRjQYq2aEgOE2mUb2kUY/uxIPy8t2g6pgD/CGABGOKTL0NXohHmn1N5uLBTGvLH8phEseUwz2UUR78SsqRbmMsvMFknywB8uMsWN0QSITbMYFCbUfblORf244avaOxSEd0bFf4qN2WxDK9u+ANv+P+az0dHhNRPmr4FdI2iSThgU+acUwxSOCUxDnP8JnQ2+IwxqhWgJ+pQs4oOQ1yF6YD3dHiEj31gkeaJJrH7vjqK0o5IaxFEOpsGJfyOLp8QUI+h8UjGmwhxoyMByzDcwzSwLsJVtrftCFE/ZD80o6EqDWLZ6C1STG1W4hqs9IY+dKHAdVsypPrnO04z4k/sIVQoxlogqAh0ItzSWQdMARBXHR0IdiWj+tcBLf2GqDplgpUCSD142aS4Ej1woCycmAtjKwNIEJIufAYAK7kQCSS2EjF/0B2XwRnEgISH/1ZQPg8W2zAhYafxj8G8kEpqtWIaxYqGiOQ1uggSUB3jITfKMV1wvQFREKga4D+ZjgM2WPdUTFmuBnVcAcR9J7SjkiNyuIJcnLMdMPSwQzSlKKLZsLfRw2PLwkG6sOs6Mqi6ArHeDR1ezgYUC/62zlGAN6ED3NCgDxB1ScPDcDktD4UP4Sb8rpmpPCXIuqREQlPzmIo6dMwFrGJFPYr9sYI5CW7cakApUsw1CtJlk9xbefKD9Sdj+7/2anKzRV0fH5zC12td280ifwuX79RXvyyETHdPsFe0BtgpyzBP/pcv79d/giXscjaGjZJFwvoRjX36hIT/SsnROlPuwDzx+hmYKTNlIzQ1/AY9b/LLzYHsrLvzp1bKF+dkUaBGmN0rc9b4CifvUz9qYAUelQtZSeBHExaD5A4J4Wo+yBNcaRRHoMXgg/yW/ZUUB813gQalK8MiZ5K/tf3339f2RUvsGWYME4dx+F/SP4/TDknkxgPBl1n/ltbmZUDgLMJsGG2w3QGrde39WaNIOREFSSxrUHVkGldOSePMaVYmh7gQlJkoncxacxi+IrDdmACycrHgUpjEvzLbAEyaeJvM+3E1eir8E61fxLOCEPQxCktRbdWARThYHCgLG+SRknzIO0EjmbLBuPe6hNdNg61S21tzBfurzqwCoSPdLmRfvS6uRm3cG4mvdyMK9I37gBDbRaLAbeDHKsvli+miRdmnaJry4/d5cX6QjDactY6JWgXSYmet6Xq1IJ76nwgQOD7x5UH3+I66B0iKx5+Vb11onx9GRdGhD5ewsgM2i6PFbW884TCrCk+xHAfGE4IlAhj0toTXJ2hpYUhbHPatQkkAw3mOFvcBkqz8Hci4P2f98+tPT1VvyoVGkUOImsIrXisVvxnK/7zXxX/iff/RDs61e1dQIDXt7ZG4+8k/lMfwcWJf9n9D693dfj3/8Rex/Hf0RntasV//kbxn+7sfRIFdPnIX46QXSf4Lw8jgtf/KFqZXxS6yQ7xLKZHeMY9xXQJ96r6X1S0penuKb5hfE+qZ//bbx5IHew51PPmoZ6Dbwm5iyOFvOrHfP7NEL7ZA5qlZ7ybKEChPPTmG7uL+aIV+hmxniF2PqMz4jTze3vaM1NORQO1mfMbjHs/DUz3Onf4UcPzhq9nCzV3s4aDhgoAbFsbA0xXcgQvJZj/paA3I9BC0KuBx4B5zofRHAIe7GpEIotG0GFoNNBl/oX0tck6APQiXIdShlBotMmaRCHTYOIQ1X594yxIpLY2AY62tp/Gx99vINv7WOJFCxV1Vopvo3iOKVILAjEm2JS4vAgMqxBQ+YhgB5AW0dfJ0/hxIkg3+AKwHcvrdlI+gP6LPDv90XbUbNFwVPQvEI/L03vVx9cwAPjxx7U75+RAJrKkkYCxocRivHJVz+vYmmodtFRrH2Sz8VQQ5W+GIg/Hdc12euycJuOJk/V1++sApHXAjyK/RYOgJXJkJf/RIf7oxB9HqDEp+PyCncQKfeeb/RJ9pPn+2c4IraVRdKhWfx80ySWJEh2J7YtI0ZHOHvy3e1+ABwB+TmRcmrdKho2+tYh/ci3rMHcmAq8FRyC6Ecx8zuH+aFwyYI5nuv09sGiA5evOFMEz/TRLZc0Tz55/LN/7xAFR51i01L5iHl1VOelVqQM91THui8aFZmTVEaWQ4adXeKerihzr83TA51zI0N5Qh7Pv6qALYcMDuTwwjbSDLnkRr7PfmzzZl4qJyVyvaonLl9yxbNDRamM4gSIfl+vPV6F+CcGHu16rxFnvtcWa2rhysnCl5/OkhTQvQwib9hYXg7WRNQioiFEYX3sbvbJ0lz1ZktBV9GoXNCc9APwQ39U+2hEZi+N5i2mhJQJFE+g4CrjYpvkaG8Qlt1h9zB0WbKzSgCGBZNVGiK86cIwRFGisnEsuZEeHxLFhPuJzJgXQE5+3E1hvk8JUFuCA35/rH3CkN61cRupJp2F88xMG6idK4FCouG5MYUvN8eMhEQR6v64QOOLrOiURg7iSouKLYiEZNszLN4XL6/s5fTlBCuApKP2WZg7YyWgv6UoyTTx+acavkBIOHvHS7EEC4kCOxjd0sBKElQzORMHW8RRhO/5CBy3k4vIDZ/lQXXyhkzOExAJ1lHKO/quyaTQW6RrD80kVtS0MjC2uhuOqaBCAvD5EY6UAE/ULoPwZEFhugM44Xt4O4YjStjYqDCO01gQeOkHeIlJnuOGMN9YyjJRGlAyoZHs2Tjt4plGD+qru3nvgyN5DwpkESB8BDbjM2N7WS09pzRt1HOizCO+WRyeFDJAEDA/QEd4AWaE72CHfFx2oWaa1hzcE5GjmtaT60/gElGn7Z6A5UCr06VYDNHXVNwcpz1dsf0bb4Tre8aUaUSq0IZ2dRdCgTpL5bX2F8iWnvGZsB4pyvUIoc3Zj6tfP4Z0XxVuYAggCYkIB3cDTDAA4XwsGHVgQJ1bTEW39bK2IlOqoa7VJfJDYdEe9orq98eM6IGyLYuP8X79YI07E6JI69mj5f1v+39/v/v/Xo92vv67Gtnd3b2/d//S78f+mMdT/13P/vmj/fzTW3cX3/3fHtsbQ/xvtat3//lv5f98sFvvBzGIbPs5h4EbtzsXa5TPlBxOV26dwk/il85X5E/T2INyit3jRnV6qXfmucud+5cbdyhfXvKNoq9/cqk48wfXkc49I6MfPdRv/l100muz+Bwm1wX3Dwe39dDkD9Dbm7Y1I3gEX6/iB8aClpr5gek0QPyor7h1DnErhwZGpVP0FPuQ6HWCofmyU3s5jO+zViYonAkZIn1R2arBQMR4xzCrGy15IVabQhFHEWyDEwuKZjZhfeiUhANKwnW69T/6BseG6qNMmfcKwEZoUDeruCF67kG+H1EG8Kya81dlO1GkWxB2rG2oRqmWXlwpVi+cdeofcWpL0j7nvpVEC2JgkjZrxLhJrIkm4Zf3hd5Ufz9YmzkijUOOWLdLW6JjkTp8gP1+FX+SAKany9Sc0dEg3Mgk0T7L5kj3ANkSwUGotq5NwLsVu4nJmrkBu9WxOHj26Jd62S96x8/hRu/e1zWD0pDCM1PMIpeRwMr412huIq6Et4X1V5OpseiYY2Y1GWkO296J06eik8U/QUWHL+SkxKgZP8yCBWXQclz+dqH1wErd8Pr2y9uiU+xQP/mVbdb8+i+cksA3UXpCuxm/WFoARu6yp6CujmAHrQlNxU8RYSgRPxeEt+4gc4ncT2UKlMDQ2iHtlsbH0cLvjfsBrACV0S27lwXz586u1D58wgUX7vXAOvtL4NHd+FvfUjp+i0Lk3nwrdBcAyOatJn/md5WKUlM8TCGuSxUr1MhOjGRnVYSuHpxmDIaIgVtRMqWDafpjmqIw1yXHS+Ygk+xYrpIEF6F3mRQ/HRZzIcY4dYduJbtgodDU7ncvxk2lzRgY6ADZoBC8Jy7ZvYxaYwvq8RfJM15THTqoz4sgBsNm6sbdY9vOqs611q6NLWfXV8e8NpjtlEA/p7HLwjdrGgtwPjm4mxlMe0pqVFmOv1EImCDuroQ5iAWrRl87zrgNmoCERUn+fExRhY4gOH8JdwTFEoxv5bgXk4d64wNcRj3XXYe/IxjzNN/yYxC1BoqJfri8R6SUoQxY3+M1ZYb4hKHAYf0o4EjMgHzCAnk74x6jY9g+vxN7yAFkeIEwnj/++duZriUqqteVFXJxsQAtiYowRlpyr6V2bio6NetTTkEnxIEr0hflf+JmS3AmJ4AVBOmr84+qk7znCEzSm5xEyclVUYB2SxIDTELu6qFpAMWb3XT3reJHGaLCfd5ItFZS18w+lplGDDA3YHdxh0dbGhJSPBZmQRkTgWFCqbiD8I0zmUdmOYdGTeOeFMLHN1iaeuZOnaVQ9aqr3HpDDjh5Wx2fdydu1D2/XJlbcm9+yU4jJeAvchpMN8IngQ+M0Fe/sQRCEe3i8XdJ1Kggw0dnPQIH+onLuWvBslunK3bs0ahrS15ZPVVdngjf5zFZ++EiKPX/yeSeqJLV5vFlVkCF+D/yj2uo2inCc+XwkAEZZSYSJcpO45vlicrDG6gfO2QmJTrSgYQUmWj7LepMrMi+l/1hwxZ+0waUiNibc6iiyEFs8RT9nsylVPKY660mP+mNm90GpA0VnX7FkZOhps1m5Nn4JtAAKd+2Dj8rffcJMnokzGPw+Q/Y0TS/BYKBigJ6FhRHowJn0ygZ6T0on6xh6T8msjl2wlayKW//pVMFniHWOgFWFWzaT4qzfi/vwklQr8G/V5FN/OLjP72X2bDbuW9vIu0yOJeZC5KW2qHnZ19uhRk7N14dT68DMLlMQBb2t47H7dmKbcIC7cO5M7cb9ys0VevoMizgmF/uRI9Dulz9ZLJ/Dw7XcM9foNbjlC4/Wnj7DM8QWTpcv3q8uLAn6HzEV3z54zBlA5Sxnm3ntmH9MCDkYmAdJATXo98ZzDviZyJ5gEQ5Hrmfd9Y8u4LUEzgUPbCBTvOUJfsg1kra0DlmD1MTNaf5KdImt8JJjwuMxXIcMbpMcYfs4R5gDn0jouQvV1Q9hvsBdH+z0n1lIcb96QtErHHrg7+asq4kAO8IPwhYc+BS5Ckc3jeGxxzAufRrvFSWk9mcqcjXQGNlh2Cy0B1JZZE8yvi2K19cEbGiK65xNmLzxQBo8wZ8ZpKxKqU0iF7KSQ4tzBg964l9f41wbphkDtfGu9fgMkdTifbj2qzl6ArLzgQ06GapiAT0DBlrO6KeaD72Dme6PtnS+S52d1MBv0KkT5eys7gZGLOBS0uhgXBoKBDYMRqQhpA5rVwXttwACFpE2RIx8P3vw1ysJaXDspaSMwJQABRFxXlJEqpMoZPWS9zTedDZmVyc1zMl4O8Avml99/bF+lmVjOO/bEwQLBVPcjegErzeVj7OoNEjT7bRm6sogRvZZsrLrlaOZMI08GyQDne5uLphsOOoG0nkIhj49jyIcbtYfaJH4IdjoLYCFaCb5cng03Fs3q/8c7aypOoB6BugB5Ss/ANp/1nzf896et99N7f0LHoenyGrBJGFiaqFLI3+HtSH62+wifzV6MbKazbOXYn8//WuWbFpC7yvU3xAEqQVaX6FI69MK5PAFtbPfZO2xdgaHyJxKPYCoM7yBcSS6f6EsKjdkmyK5j2mxMj5XvfVEYs5U0TmK6urjz2sTy54Dgs0yzd2JGPZR5EdSsCvRfV+H50MsGinIraedhHAxo3DUTM48NpzLwACy8W6r4aCbDVvgph++h4Of08MZ/pWJeOpbJD7bjPgBD6kC+iosKMWrvM4jSgDVM3FSNNiTYE6vV3jeKn8PZkE+TOX7cA4ZVt868s5+pQ56VHNy5Cj/YfUwqYBGHcW6cKvnMeDIxLC6n7wogCJnICHHotFXxWlArEct9gHZh3RCHBU3SBr9Osh3sqEzIZPzCOS6oiUz1ecYBII3QMwVDSWjU6UGL6KU/3H3hEQ3Urrf3pTXhyoWNUcawKJ1I6bS+Vx6kA/tFLsPPtVfVDgRmSlfX4OlZ/F+pQ1B/O9r5z5i0pMe1roBmB3NwBQaeSlY68vjprKSvS6R+4oj5Nuf3yiOKEnIQ/4KCIo0gBEJ8k/vxscBiSwQEUHqrYMk0AlhTFt6no1nk/J90E/MIzLQSjb9uCT86Y0i9ICVp/GSIVGe4A0SDef8WELLDCDQSlnD8VC9/58IAKbFJ4GZ2ziv2L1BdQcFwTqjmkqJTFPuBL0Gw/1IBqJfNlTRcPky0SbI3ZVkosugemER2RJQQAGZuBNcMj3RAwgmVhmo1sO6pYgTIdOo3z3c7JK/OlajJ4sAAFl5h21quK0Rw5k2k2iT+KZ0tHN7R9/mnewSEnIBSsCy1cd2bMFyO+Wm3RLCUIum0+RwAeJDI7kblmxY/8SAVLVpQGp9wJ1XnuOqsQwC4129xez5UZN5uyKMPcP1R+2RSu1SNpsb4WhH8L1JvLGdDBX5sgQqAmgI1L4mpwL6tjyMA261B47y2BDeMw88eEexkaZQBwR6yfgFIt2bDUx66niCkhEIl1RkdBIs3vTYgR2yvDjvp0xf8GZ/dtKBGP0baIHzYePs+E/MSFzqiVzex1kcVN7+nNGe17NOfJs5snkncMEH0iitHuVYJgzM3bdTVFlYq0zSpEUhgxeLpQeSsqEPy72husjmoQ0XAU2udwwFbLL1GNgTwOZG18sFdBBUil8oAoCxdmRyQxw/pkbuS45vNUcknJP/iHfAog5tZOKb9G3Zbn37H5mYiPV16x3RP77UyQZ95AbVdkvL5Eo2Vr55J9OfQVksn50CqbKjb6dABpOSYUeftfOlGqB+ObB7qF+u6XIX3jD8aLWyMkv9WGvLp2ge6tCCDDu2ACZ2yuGm68iejhYPNZ1Ave9IoVa0TetpPa2n9bSe1tN6Wk/raT2tp/W0ntbTelpP62k9raf1tJ7W03paT+tpPa2n9bSe1tN6Wk/raT3/s8//BYfJwwUAyAAA"
tarfile.open(fileobj=io.BytesIO(base64.b64decode(_CODE)), mode="r:gz").extractall(APP_DIR)
if str(APP_DIR) not in sys.path:
    sys.path.insert(0, str(APP_DIR))

# ---- 設定 ----
MODELS = Path("/content/models")
os.environ.update(
    MN_MODELS_DIR=str(MODELS), MN_DATA_DIR="/content/data", MN_THREADS=str(os.cpu_count() or 2),
    MN_ASR_DEVICE="cuda" if GPU else "cpu", MN_ASR_COMPUTE="float16" if GPU else "int8",
    MN_LLM_BACKEND="transformers",
    MN_LLM_HF_ID="Qwen/Qwen3-4B-Instruct-2507" if GPU else "Qwen/Qwen3-1.7B",
)

# 預先載入 cuDNN，避免語音模型找不到 GPU 函式庫
if GPU:
    try:
        import nvidia.cudnn
        for _f in sorted(glob.glob(os.path.join(list(nvidia.cudnn.__path__)[0], "lib", "libcudnn*.so*"))):
            try:
                ctypes.CDLL(_f, mode=ctypes.RTLD_GLOBAL)
            except OSError:
                pass
    except Exception:
        pass

# ---- 下載模型（從 Hugging Face，約 3.5 GB）----
print("⬇️ 下載語音辨識與講者分離模型…")
from huggingface_hub import hf_hub_download, snapshot_download
snapshot_download("SoybeanMilk/faster-whisper-Breeze-ASR-25", local_dir=MODELS / "breeze-asr-25-ct2",
                  allow_patterns=["*.json", "model.bin"])
def _get(repo, fn, dest):
    dest = MODELS / dest
    if not dest.exists():
        dest.parent.mkdir(parents=True, exist_ok=True)
        os.symlink(hf_hub_download(repo, fn), dest)
_get("csukuangfj/sherpa-onnx-pyannote-segmentation-3-0", "model.onnx", "diar/segmentation.onnx")
_get("csukuangfj/speaker-embedding-models",
     "3dspeaker_speech_eres2netv2_sv_zh-cn_16k-common.onnx", "diar/embedding.onnx")
_get("csukuangfj/sherpa-onnx-punct-ct-transformer-zh-en-vocab272727-2024-04-12",
     "model.onnx", "punct/model.onnx")

from app import colab as mn
SELECTED = None
print(f"\n🎉 準備完成！（{int(time.time() - _t0)} 秒）請繼續執行「步驟 2」選擇錄音檔。")

In [ ]:
#@title 步驟 2：選擇錄音檔
#@markdown **從雲端硬碟選擇**：執行後下方會出現雲端硬碟的資料夾，點資料夾進入、點錄音檔選取（🎵 是錄音檔，✅ 表示已處理過）。
#@markdown 逐字稿與會議記錄會**存到錄音所在的同一個資料夾**。
#@markdown
#@markdown **從電腦上傳**：上傳的錄音與結果會存到下面指定的雲端硬碟資料夾（不存在會自動建立）。
錄音來源 = "從雲端硬碟選擇"  #@param ["從雲端硬碟選擇", "從電腦上傳"]
上傳存放資料夾 = "會議記錄"  #@param {type:"string"}

import shutil
from IPython.display import display
from google.colab import files

try:
    mn
except NameError:
    raise SystemExit("⚠️ 請先執行「步驟 1」")

def _pick(p):
    global SELECTED
    SELECTED = p

if 錄音來源 == "從雲端硬碟選擇" and USE_DRIVE:
    _browser = mn.FileBrowser(str(DRIVE_ROOT), start=str(SELECTED.parent) if SELECTED else None,
                              on_select=_pick)
    display(_browser.box)
else:
    if not USE_DRIVE and 錄音來源 == "從雲端硬碟選擇":
        print("⚠️ 未連接雲端硬碟，改為從電腦上傳。")
    _dest = DRIVE_ROOT / 上傳存放資料夾.strip().strip("/") if USE_DRIVE else DRIVE_ROOT
    _dest.mkdir(parents=True, exist_ok=True)
    print("請點下方「選擇檔案」上傳錄音（大檔案上傳需要一些時間）：")
    _up = files.upload()
    if not _up:
        raise SystemExit("沒有選擇檔案")
    _name = next(iter(_up))
    SELECTED = _dest / _name
    shutil.move(_name, SELECTED)
    print(f"✅ 已上傳並存到：{str(SELECTED).replace(str(DRIVE_ROOT), '我的雲端硬碟', 1)}")
    print("請繼續執行「步驟 3」。")

In [ ]:
#@title 步驟 3：產生逐字稿與會議記錄
#@markdown **會議資訊**（可留空，會議名稱預設為錄音檔名；之後也能在 Word 檔中修改）
會議名稱 = ""  #@param {type:"string"}
會議日期 = ""  #@param {type:"date"}
地點 = ""  #@param {type:"string"}
主席 = ""  #@param {type:"string"}
#@markdown **辨識設定**（填入正確的講者人數，分辨講者會更準確）
講者人數 = "自動判斷"  #@param ["自動判斷", "2", "3", "4", "5", "6", "7", "8", "9", "10"]
專有名詞 = ""  #@param {type:"string"}
#@markdown ↑ 人名、公司、產品、術語，用逗號分隔，例如：聯發科, Kubernetes, 王小明
產生會議記錄 = True  #@param {type:"boolean"}
下載Word到電腦 = False  #@param {type:"boolean"}

from IPython.display import Markdown, display
from google.colab import files

if not SELECTED or not Path(SELECTED).exists():
    raise SystemExit("⚠️ 請先在「步驟 2」選擇錄音檔")

meta = dict(title=會議名稱.strip() or Path(SELECTED).stem, date=會議日期, location=地點, chair=主席,
            num_speakers=None if 講者人數 == "自動判斷" else int(講者人數), hotwords=專有名詞.strip())
TR, MINUTES, META = mn.process(str(SELECTED), meta, 產生會議記錄)

display(Markdown("---\n## 🔊 各講者聲音片段（用來辨認是誰，可到步驟 4 填上姓名）"))
mn.preview_speakers(TR, str(SELECTED))
if MINUTES:
    display(Markdown("---\n" + MINUTES))
if 下載Word到電腦 or not USE_DRIVE:
    for _f in mn.output_files(SELECTED, META):
        files.download(str(_f))

In [ ]:
#@title 步驟 4（選用）：替講者填上真實姓名
#@markdown 參考步驟 3 的「各講者聲音片段」，填入姓名（不需要的留空）。逐字稿與會議記錄都會更新。
#@markdown 要修改**較早處理過**的會議：先在步驟 2 選擇那個錄音（標示 ✅已處理），再執行本步驟。
講者_1 = ""  #@param {type:"string"}
講者_2 = ""  #@param {type:"string"}
講者_3 = ""  #@param {type:"string"}
講者_4 = ""  #@param {type:"string"}
講者_5 = ""  #@param {type:"string"}
講者_6 = ""  #@param {type:"string"}
講者_7 = ""  #@param {type:"string"}
講者_8 = ""  #@param {type:"string"}
#@markdown 勾選後會依新姓名**重新撰寫**會議記錄（約 2～5 分鐘）；不勾選只替換名稱（立即完成）。
重新產生會議記錄 = False  #@param {type:"boolean"}
下載Word到電腦 = False  #@param {type:"boolean"}

from IPython.display import Markdown, display
from google.colab import files

if not SELECTED:
    raise SystemExit("⚠️ 請先在「步驟 2」選擇錄音檔")
_names = [講者_1, 講者_2, 講者_3, 講者_4, 講者_5, 講者_6, 講者_7, 講者_8]
_mapping = {f"講者 {i}": n for i, n in enumerate(_names, 1) if n.strip()}
TR, MINUTES, META = mn.rename(SELECTED, _mapping, 重新產生會議記錄)
if MINUTES:
    display(Markdown("---\n" + MINUTES))
if 下載Word到電腦 or not USE_DRIVE:
    for _f in mn.output_files(SELECTED, META):
        files.download(str(_f))

---
### ❓ 常見問題
- **出現「無法連線到 GPU 後端」或「已達使用上限」**：免費 GPU 額度暫時用完，請過幾小時或隔天再試；也可以不用 GPU 執行（較慢）。
- **講者分得太多或太少**：在步驟 2 選擇正確的「講者人數」再執行一次。
- **專有名詞辨識錯誤**：在步驟 2 的「專有名詞」欄填入正確寫法。
- **結果檔案在哪裡**：和錄音檔同一個資料夾，檔名以會議名稱開頭：`會議記錄.docx`／`.md`、`逐字稿.docx`／`.txt`／`.srt`，另有一個 `_處理資料.json`（供改講者姓名用，請勿刪除）。
- **雲端硬碟授權失敗（credential propagation was unsuccessful）**：重新執行步驟 1，授權頁面點「全部選取」勾選所有權限；或確認瀏覽器沒有封鎖彈出式視窗。
- **資料夾清單沒看到剛放進去的檔案**：按清單上方的「🔄 重新整理」。
- **處理到一半中斷**：逐字稿完成時就會先存檔；重新執行步驟 1、在步驟 2 選同一個錄音，再用步驟 4 勾選「重新產生會議記錄」即可補產生。